# Stroke Prediction — Baselines

**Goal:** Establish baseline classifier performance using three strategies:
1. No resampling (raw imbalanced data)
2. Class weights (penalise majority class)
3. SMOTE (synthetic oversampling of minority class)

**Classifiers:** Logistic Regression, Random Forest  
**Metrics:** F1, PR-AUC, MCC (accuracy is misleading on imbalanced data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, average_precision_score, matthews_corrcoef,
    classification_report, PrecisionRecallDisplay
)
from imblearn.over_sampling import SMOTENC

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

DATA_PATH = Path('../data/healthcare-dataset-stroke-data.csv')
RANDOM_STATE = 42

## 1. Load & Preprocess

In [ ]:
df = pd.read_csv(DATA_PATH)

# Drop id — no predictive value
df = df.drop(columns=['id'])

# Drop 'Other' gender — only 1 record, causes issues with stratification
df = df[df['gender'] != 'Other'].copy()

# Impute missing bmi with median (calculated on full dataset before split)
bmi_median = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(bmi_median)

print(f'Shape after cleaning: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Stroke rate: {df["stroke"].mean()*100:.2f}%')

In [ ]:
NUMERICAL   = ['age', 'avg_glucose_level', 'bmi']
CATEGORICAL = ['hypertension', 'heart_disease', 'gender', 'ever_married',
               'work_type', 'Residence_type', 'smoking_status']

# OrdinalEncoder: converts string categories to integers
enc = OrdinalEncoder()
df[CATEGORICAL] = enc.fit_transform(df[CATEGORICAL])

# Feature matrix: numericals first, then categoricals
feature_cols = NUMERICAL + CATEGORICAL
X = df[feature_cols].values
y = df['stroke'].values

# Indices of categorical columns — needed for SMOTENC later
cat_indices = list(range(len(NUMERICAL), len(feature_cols)))  # [3, 4, 5, 6, 7, 8, 9]

print(f'Feature matrix shape: {X.shape}')
print(f'Categorical indices: {cat_indices}')
print(f'\nFeature order: {feature_cols}')

## 2. Train / Test Split

Stratified split: preserves the stroke/no-stroke ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale numerical features only (fit on train, transform both)
# Categoricals stay as OrdinalEncoded integers
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[:, :len(NUMERICAL)])
X_test_num  = scaler.transform(X_test[:, :len(NUMERICAL)])

X_train_s = np.hstack([X_train_num, X_train[:, len(NUMERICAL):]])
X_test_s  = np.hstack([X_test_num,  X_test[:, len(NUMERICAL):]])

print(f'Train: {X_train_s.shape} | Stroke cases: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test:  {X_test_s.shape}  | Stroke cases: {y_test.sum()} ({y_test.mean()*100:.1f}%)')

## 3. Evaluation Helper

For each model we compute three metrics:
- **F1**: harmonic mean of precision & recall
- **PR-AUC**: area under precision-recall curve — robust on imbalanced data
- **MCC**: Matthews Correlation Coefficient — best overall measure for imbalanced datasets

In [ ]:
results = []

def evaluate(method, clf_name, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]

    f1     = f1_score(y_te, y_pred)
    pr_auc = average_precision_score(y_te, y_prob)
    mcc    = matthews_corrcoef(y_te, y_pred)

    print(f'\n[{method}] {clf_name}')
    print(f'  F1: {f1:.4f} | PR-AUC: {pr_auc:.4f} | MCC: {mcc:.4f}')
    print(classification_report(y_te, y_pred, target_names=['No Stroke', 'Stroke']))

    results.append({'Method': method, 'Classifier': clf_name,
                    'F1': round(f1, 4), 'PR-AUC': round(pr_auc, 4), 'MCC': round(mcc, 4)})
    return clf

## 4. Baseline 1 — No Resampling

Model trained on raw imbalanced data. We expect high accuracy but poor F1 for the minority class (stroke).

In [ ]:
evaluate('No Resampling', 'Logistic Regression',
         LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
         X_train_s, y_train, X_test_s, y_test)

evaluate('No Resampling', 'Random Forest',
         RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
         X_train_s, y_train, X_test_s, y_test)

## 5. Baseline 2 — Class Weights

`class_weight='balanced'` tells the classifier to **penalise errors** on the minority class more heavily. The data does not change — only the loss function.

In [ ]:
evaluate('Class Weights', 'Logistic Regression',
         LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
         X_train_s, y_train, X_test_s, y_test)

evaluate('Class Weights', 'Random Forest',
         RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE),
         X_train_s, y_train, X_test_s, y_test)

## 6. Baseline 3 — SMOTE

**SMOTENC** (SMOTE for Nominal and Continuous): generates new synthetic minority samples via interpolation while correctly handling categorical features (no interpolation between categories — picks the most frequent value from neighbours).

Applied **only to the training set** — the test set remains untouched.

In [ ]:
smotenc = SMOTENC(categorical_features=cat_indices, random_state=RANDOM_STATE)
X_train_res, y_train_res = smotenc.fit_resample(X_train_s, y_train)

print(f'Before SMOTE: {X_train_s.shape} | Stroke: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'After SMOTE:  {X_train_res.shape} | Stroke: {y_train_res.sum()} ({y_train_res.mean()*100:.1f}%)')

In [ ]:
evaluate('SMOTE', 'Logistic Regression',
         LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
         X_train_res, y_train_res, X_test_s, y_test)

evaluate('SMOTE', 'Random Forest',
         RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
         X_train_res, y_train_res, X_test_s, y_test)

## 7. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
metrics = ['F1', 'PR-AUC', 'MCC']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, metrics):
    for clf_name, grp in results_df.groupby('Classifier'):
        ax.bar(
            [f"{r['Method']}" for _, r in grp.iterrows()],
            grp[metric].values,
            label=clf_name, alpha=0.8
        )
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=20)
    ax.legend(fontsize=8)

plt.suptitle('Baseline Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/baseline_results.png', dpi=150, bbox_inches='tight')
plt.show()